In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim, upper, current_timestamp, coalesce, lit, date_format
from pyspark.sql.types import IntegerType, DoubleType, StringType

StatementMeta(sparkpoolbank, 78, 2, Finished, Available, Finished, False)

In [5]:
bronze_path = "abfss://bronze@stbankingretail.dfs.core.windows.net/Account_Master/*.csv"

df_bronze = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(bronze_path)

# df_bronze.printSchema()

StatementMeta(sparkpoolbank, 78, 6, Finished, Available, Finished, False)

In [6]:
from pyspark.sql.functions import col, current_timestamp, date_format, lit, trim, when
from pyspark.sql.types import DoubleType, StringType

# 1. Spark Session  12-byte INT96 Timestamps Force 
spark.conf.set("spark.sql.parquet.outputTimestampType", "INT96")
spark.conf.set("spark.sql.parquet.writelegacyformat", "true")

# 2. Data Cleaning Transformations
df_clean = (
    df_bronze.withColumn("Account_Number", trim(col("Account_Number")))
    .withColumn("Customer_ID", trim(col("Customer_ID")))
    .withColumn(
        "Account_Type",
        when(
            upper(trim(col("Account_Type"))).isin("CURR", "CURRENT"),
            lit("Current"),
        )
        .when(
            upper(trim(col("Account_Type"))).isin("SAVINGS", "SAV"),
            lit("Savings"),
        )
        .otherwise(lit("Unknown")),
    )
    .withColumn("Current_Balance", col("Current_Balance").cast(DoubleType()))
    .withColumn(
        "Status",
        when(col("Status").isNotNull(), trim(col("Status"))).otherwise(
            lit("Unknown")
        ),
    )
    # Important: IngestionDate Explicitly String Cast
    .withColumn(
        "IngestionDate",
        date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss").cast(
            StringType()
        ),
    )
)

# 3. Select final columns
df_final = df_clean.select(
    "Account_Number",
    "Customer_ID",
    "Account_Type",
    "Current_Balance",
    "Status",
    "IngestionDate",
)

silver_path = (
    "abfss://silver@stbankingretail.dfs.core.windows.net/Account_Master"
)

# 4. Save to Silver as Parquet
df_final.coalesce(1).write.format("parquet").option(
    "parquet.writer.version", "v1"
).mode("overwrite").option("overwriteSchema", "true").save(silver_path)

StatementMeta(sparkpoolbank, 78, 7, Finished, Available, Finished, False)

In [4]:
# df_final.count()

StatementMeta(sparkpoolbank, 78, 5, Finished, Available, Finished, False)